In [ ]:
library(Signac)
library(Seurat)
library(ggplot2)
library(GenomicRanges)
library(scRNAtoolVis)

In [ ]:
scmul_3 <- readRDS("02_processed_data/Pri3/Pri3_RNA&ATAC.rds")
merged_scmul <- readRDS("02_processed_data/Combined_samples/Combined_All3.rds")

In [ ]:
DefaultAssay(scmul_3 ) <- "RNA"
Idents(scmul_3) <- "RNA_Celltype"
marker_list <- list(
  "Tumor epithelial cells" = c("EPCAM", "KRT8", "CFTR"),
  "Macrophages" = c("CD163", "CSF1R", "MERTK"),
  "T cells" = c("CD3D", "CD3E", "CD8A"),
  "Normal epithelial cells" = c("INS", "GCG"),
  "Fibroblasts" = c("PDGFRA", "LUM", "DCN", "COL1A1"),
  "Plasma cells" = c("MZB1", "XBP1", "PRDM1"),
  "Mast cells" = c("TPSAB1", "KIT")
)

marker_list_in_data <- lapply(marker_list, function(g) g[g %in% rownames(scmul_3)])
marker_list_in_data <- marker_list_in_data[sapply(marker_list_in_data, length) > 0]
generate_colors <- colorRampPalette(c("#483D8B","#00FFFF","#F8F8FF","#FF69B4","#8B008B"))
colors <- generate_colors(100)
DotPlot(
  scmul_3,
  features = unlist(marker_list_in_data, use.names = FALSE)
) +
  scale_color_gradientn(colors = colors) +
  RotatedAxis()

In [ ]:
custom_colors2 <- c("#0675b4","#8DBCCA","#03a076","#E12021","#efe447","#F8B9AB","#D56B19", "#B288AB", "#121014")
clusterCornerAxes(
  object = scmul_3,
  reduction = "umap",
  clusterCol = "RNA_Celltype",
  cellLabel = TRUE,
  pSize = 0.2,
  cellLabelSize = 6,
  noSplit = TRUE
) +
  scale_color_manual(values = alpha(custom_colors2, 0.6)) +
  scale_fill_manual(values = alpha(custom_colors2, 0.6)) + 
  theme(text = element_text(family = "sans"),
        legend.title = element_text(face = "bold",size = 18),
        legend.text = element_text(face = "bold", size = 18)
  )

## FBXW2 CoveragePlot

In [ ]:
DefaultAssay(scmul_3) <- "ATAC"
Idents(scmul_3) <- "RNA_Celltype"
idents.plot <- c("Tumor epithelial cells", "Normal epithelial cells")

p_TvN <- CoveragePlot(
  object = scmul_3,
  region = "FBXW2",
  features = "FBXW2",
  expression.assay = "RNA",
  idents = idents.plot
) &
  scale_fill_manual(values = c("#D39300", "#DB72FA")) &
  theme(
    text = element_text(family = "sans", face = "bold"),
    plot.title = element_text(size = 14, face = "bold"),
    axis.title = element_text(size = 12, face = "bold"),
    axis.text = element_text(size = 12, face = "bold"),
    legend.text = element_text(size = 12, face = "bold")
  )

p_TvN

In [ ]:
FBXW2_promoter_Find <- GRanges(
  seqnames = "chr9",
  ranges = IRanges(start = 120792300, end = 120792700),
  strand = "-"
)

p_TvN_zoom <- CoveragePlot(
  object = scmul_3,
  region = FBXW2_promoter_Find,
  features = "FBXW2",
  expression.assay = "RNA",
  idents = idents.plot,
  extend.upstream = 5000,
  extend.downstream = 10000
) &
  scale_fill_manual(values = c("#D39300", "#DB72FA")) &
  theme(
    text = element_text(family = "sans", face = "bold"),
    plot.title = element_text(size = 10, face = "bold"),
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.text = element_text(size = 10, face = "bold")
  )

p_TvN_zoom


In [ ]:
DefaultAssay(merged_scmul) <- "RNA"
sample_ids <- sapply(strsplit(rownames(merged_scmul@meta.data), "_"), function(x) x[1])
merged_scmul$sample <- sample_ids
tumor_cells <- subset(merged_scmul, subset = RNA_Celltype == "Tumor epithelial cells")

In [ ]:
frag_path_pri1 <- "Pri1/atac/fragments.tsv.gz"
frag_path_pri2 <- "Pri2/atac/fragments.tsv.gz"
frag_path_pri3 <- "Pri3/atac/fragments.tsv.gz"
cells_pri1 <- colnames(tumor_cells)[tumor_cells$sample == "Pri1"]
cells_pri2 <- colnames(tumor_cells)[tumor_cells$sample == "Pri2"]
cells_pri3 <- colnames(tumor_cells)[tumor_cells$sample == "Pri3"]
cells_pri1_clean <- sub("^Pri1_", "", cells_pri1)
cells_pri2_clean <- sub("^Pri2_", "", cells_pri2)
cells_pri3_clean <- sub("^Pri3_", "", cells_pri3)

map_pri1 <- cells_pri1_clean; names(map_pri1) <- cells_pri1
map_pri2 <- cells_pri2_clean; names(map_pri2) <- cells_pri2
map_pri3 <- cells_pri3_clean; names(map_pri3) <- cells_pri3

frag1 <- CreateFragmentObject(path = frag_path_pri1, cells = map_pri1, validate.fragments = TRUE)
frag2 <- CreateFragmentObject(path = frag_path_pri2, cells = map_pri2, validate.fragments = TRUE)
frag3 <- CreateFragmentObject(path = frag_path_pri3, cells = map_pri3, validate.fragments = TRUE)

Fragments(tumor_cells) <- NULL
Fragments(tumor_cells) <- list(frag1, frag2, frag3)


In [ ]:
Idents(tumor_cells) <- "sample"
DefaultAssay(tumor_cells) <- "ATAC"
p_sample_full <- CoveragePlot(
  object = tumor_cells,
  region = "FBXW2",
  features = "FBXW2",
  expression.assay = "RNA",
  idents = c("Pri1", "Pri2", "Pri3")
) &
  theme(
    text = element_text(family = "sans", face = "bold"),
    plot.title = element_text(size = 10, face = "bold"),
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.text = element_text(size = 10, face = "bold")
  )
p_sample_zoom <- CoveragePlot(
  object = tumor_cells,
  region = FBXW2_promoter_Find,
  features = "FBXW2",
  expression.assay = "RNA",
  idents = c("Pri1", "Pri2", "Pri3"),
  extend.upstream = 5000,
  extend.downstream = 10000
) &
  theme(
    text = element_text(family = "sans", face = "bold"),
    plot.title = element_text(size = 10, face = "bold"),
    axis.title = element_text(size = 10, face = "bold"),
    axis.text = element_text(size = 10, face = "bold"),
    legend.text = element_text(size = 10, face = "bold")
  )

p_sample_full
p_sample_zoom


In [2]:
sessionInfo()

R version 4.4.1 (2024-06-14)
Platform: x86_64-pc-linux-gnu
Running under: Ubuntu 20.04.6 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas-pthread/libblas.so.3 
LAPACK: /usr/lib/x86_64-linux-gnu/openblas-pthread/liblapack.so.3;  LAPACK version 3.9.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C               LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8    LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C             LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Asia/Shanghai
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods   base     

other attached packages:
 [1] scRNAtoolVis_0.1.0   lubridate_1.9.3      forcats_1.0.0        stringr_1.5.1        dplyr_1.2.0          purrr_1.2.1         
 [7] readr_2.1.5          tidyr_1.3.2          tibble_3.3